<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/11_3_RAG_Fundamentals__OpenAI_embedding_%26_Chroma_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**

1.   This notebook is an enhancement of the RAG fundamentals notebook
2.   I use an anonymized version of Tale of Two Cities as the knowledge base
3.   Chunking is done using a function I’ve written
4.   Embeddings are created using OpenAI model text-embedding-3-small
5.   Embeddings are stored in Chroma DB
6.   LLM used is gpt-4.1-mini
7.   It does not use LangChain or LlamaIndex


**To ensure pure grounded RAG, the following is done -**


1.   Similarity threshold - This is the most important safeguard.
2.   Strict prompt instruction - Even if retrieval returns weak matches, the LLM must refuse to invent answers.
3.   Show source chunks - This makes system auditable.




In [ ]:
!pip install -q chromadb openai
# This will throw some errors but we can ignore them because
# we're not using google-adk & opentelemetry exporters

In [ ]:
#Imports
import sys
from google.colab import drive
from openai import OpenAI
from google.colab import userdata
import chromadb

In [ ]:
#Setup an LLM that can answer questions based on Context

#define constants
MODEL = "gpt-4.1-mini"
#MODEL = "gpt-5.2"

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

In [ ]:
#Function to call ChatGPT

def message_gpt(system_message,prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    completion = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    return completion.choices[0].message.content

In [ ]:
# import a re-use function to read text from file stored in Google Drive
drive.mount('/content/drive', force_remount=True)
UTIL_PATH = "/content/drive/MyDrive/Colab Notebooks"
sys.path.insert(0, UTIL_PATH)
from file_utils_colab import read_text_from_file


In [ ]:
# Let's use an anonymized and abridged version of Tale of Two Cities for RAG knowledge base
KB_text = read_text_from_file(
    folder_path="Files/Knowledge-Base",
    file_name= "Anonymized by OpenAI_TOTC.txt"
)

In [ ]:
print(f"The number of characters are : {len(KB_text)}")

number_of_words = len(KB_text.split())
# Divides a string into a list of substrings based on a specified separator (default is whitespace) and then counts length of list

print(f"Number of words is : {number_of_words}")
#print(KB_text)

In [ ]:
# Step 1 : Chunking the document
# the chunk size and overlap is different from the other example
# where I used FAISS and SentenceTransformer
# based on trial and error and work better than the options suggested by
# ChatGPT

def chunk_text(text, chunk_size=1200, overlap=300):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = chunk_text(KB_text) #,chunk_size=50, overlap=10)
print(f"Total number of chunks: {len(chunks)}")
print("-----------------------------------------------------------")
print(f"Chunk 1 is : {chunks[0]}")
print("-----------------------------------------------------------")
print(f"Chunk 2 is : {chunks[1]}") # just to see the overlap

In [ ]:
# Step 2 : Create embeddings using OpenAI
response = openai.embeddings.create(
    model="text-embedding-3-small",   # or text-embedding-3-large
    input=chunks
)

embeddings = [e.embedding for e in response.data]

In [ ]:
# Tell LLM to answwer only with Context
# Strict prompt instruction

system_message = """You are a helpful assistant who will answer questions only based in the context that is shared with you.
If the question is about something that is not in the context given to you then you'll respond with DO-NOT-KNOW .
Do not lie or make things up.
"""
context = ""

In [ ]:
# Step 3 : Store embeddings in Chroma DB

chroma_client = chromadb.Client()

#collection = chroma_client.create_collection(name="rag_collection")

collection = chroma_client.create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}
    # distance will be behave like FAISS : distance = 1 - cosine_similarity
    # So: 0 = perfect match | smaller = better
)

# Store embeddings + text together
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(chunks))]
)

In [ ]:
#Step 4 :
# Step 4.1 : Convert question into vector
# Step 4.2 : Fetch similar vectors from Chroma DB
# Step 4.3 : Filter using a thereshold
# Step 4.4 : build context from filtered vectors and use this to make prompt
# Step 4.5 : Call LLM

def get_answer(question, threshold=0.25, debug=False):

    # Step 4.1: Create query embedding using OpenAI
    q_response = openai.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )
    q_embedding = q_response.data[0].embedding

    # Step 4.2: Query ChromaDB
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=5
    )

    retrieved_chunks = results["documents"][0]
    scores = results["distances"][0]

    # Step 4.3: Apply threshold filtering
    filtered_chunks = [
        chunk for score, chunk in zip(scores, retrieved_chunks)
        if score >= threshold
    ]

    # Debug logging
    if debug:
        print("----------------------------------------")
        for score, chunk in zip(scores, retrieved_chunks):
            print(f"Distance: {score}, Chunk: {chunk}")
            print("----------------------------------------")

    # Handle no results
    if not filtered_chunks:
        return {"answer": "DO-NOT-KNOW", "sources": "NA"}

    # Step 4.4 : Build context
    context = "\n".join(filtered_chunks)

    prompt = (
        "This is the context based on which you should answer:\n"
        + context +
        "\n\nAnd this is the question:\n"
        + question
    )

    # Step 4.5 : Call LLM
    answer = message_gpt(system_message, prompt)

    return {
        "answer": answer,
        "sources": filtered_chunks
    }

In [ ]:
print(get_answer("Who is the author of A Chronicle of Two Cities"))

In [ ]:
print(get_answer("Who is Philippe Duval married to ?"))

In [ ]:
print(get_answer("Who are the owners of the wine shop in this story ?"))

In [ ]:
print(get_answer("Who is on trial for treason against England ?"))

In [ ]:
print(get_answer("Who is the Jackal and who is the Lion ?"))

In [ ]:
print(get_answer("Why is he on trial ?")) #retains context from previous question

In [ ]:
print(get_answer("Who are the witnesses in this trial ?")) #retains context from previous question

In [ ]:
print(get_answer("What is the capital of India ?")) # can't answer since it's not in knowledge base

In [ ]:
print(get_answer("Which cities are named in this story ?"))

In [ ]:
print(get_answer("Who is the villain in this story ?"))
# threshold always does a similarity search between question and KB .But in this case similarity score is very low
# setting threshold to lower value that chunks are sent to LLM based on which it can answer this indirect question

In [ ]:
print(get_answer("Who is the resurrection man?"))
